In [1]:
from ioMicro import *

##### *Load in th parameter file for enhancers*

In [82]:
import numpy as np
import pandas as pd

h_ths = np.load('enhancer_final_params_ths.npz')['final_params']
h_ths = h_ths.astype(int)

# create an empty dataframe with hybes (rows) and ths for each icol (columns)
master_ths = pd.DataFrame(index=range(1,8), columns=h_ths[:3,1])

# fill in the dataframe with the correct ths for each hybe/icol
for h in range(7):
    i1,i2 = h*3,(h+1)*3
    master_ths.iloc[h,:] = h_ths[int(i1):int(i2),2]

In [84]:
master_ths

,0,1,2
1,121,148,181
2,221,812,445
3,244,244,221
4,330,270,221
5,221,897,200
6,148,270,148
7,221,270,148


##### *Create dictionary for all enhancer spots across FOVs*

In [85]:
# Make a yuge dict for all CRE spots across FOVs.
# 
# Use brightness thresholds loaded from master_ths (calculated separately)

CRE_counts = {}

for fov in tqdm(range(225)):
    for hybe in range(1,8):
        for icol in range(3):
            # load in all fitted spots for this particular image stack
            spots = np.load(f'Z:\\Zane_20CRE\\7_17_2024__T7_20CRE\\analysis\\Conv_zscan__{fov:03d}--H{hybe:01d}_fits_icol{icol:01d}.npz')['Xh']
            
            # correct the drift for spots as they're loaded in
            drifts = np.load(f'Z:\\Zane_20CRE\\7_17_2024__T7_20CRE\\drifts_polyA\\Conv_zscan__{fov:03d}_drift.pkl', allow_pickle=True)
            drift = drifts[f'H{hybe:01d}'][0]
            spots[:,:3]-=drift
            
            # threshold spots based on brightness values from master_ths
            h = spots[:,-1]
            keep = np.where(h > master_ths.iloc[hybe-1, icol])
            spots_ = spots[keep]
            
            # update the dictionary to contain thresholded spots for this particular image stack
            CRE_counts[f'{fov:03d}-H{hybe}-{icol}'] = spots_

100%|████████████████████████████████████████████████████████████████████████████████| 225/225 [03:00<00:00,  1.25it/s]


##### *Convert keys from CRE_counts into the actual CRE names*

In [86]:
# make list of CRE names and filter the converted dict keys into a new list, fov_counts

CREs = ('CRE001', 'CRE002', 'CRE003', 'CRE004', 'CRE005', 'CRE006', 'CRE007', 'CRE008',
        'CRE009', 'CRE010', 'CRE011', 'CRE012', 'CRE013', 'CRE014', 'CRE015', 'CRE016',
        'CRE017', 'CRE018', 'CRE019', 'CRE020')

fov_filter = [f'{fov:03d}-H1-0', f'{fov:03d}-H1-1', f'{fov:03d}-H1-2', f'{fov:03d}-H2-0', f'{fov:03d}-H2-1', f'{fov:03d}-H2-2', f'{fov:03d}-H3-0', f'{fov:03d}-H3-1', f'{fov:03d}-H3-2', f'{fov:03d}-H4-0',
              f'{fov:03d}-H4-1', f'{fov:03d}-H4-2', f'{fov:03d}-H5-0', f'{fov:03d}-H5-1', f'{fov:03d}-H5-2', f'{fov:03d}-H6-0', f'{fov:03d}-H6-1', f'{fov:03d}-H6-2', f'{fov:03d}-H7-0', f'{fov:03d}-H7-1']

filterByKey = lambda keys: {ccre_conversion[x[3:]] : CRE_counts[x] for x in keys}

ccre_conversion = {channel_id[3:] : ccre for channel_id, ccre in zip(fov_filter, CREs)}

fov_counts = []

for fov in tqdm(range(225)):
    fov_counts.append(filterByKey(fov_filter))

100%|████████████████████████████████████████████████████████████████████████████| 225/225 [00:00<00:00, 224855.47it/s]


##### *Create a cell by gene matrix for all enhancer spots using the appropriate segmentation masks*

In [90]:
# Assign spots for each CRE to masks generated from Cellpose and combine into a pandas dataframe

import pandas as pd
import numpy as np

masked_spots = []
all_cbg = []
fov_means = []

for fov in tqdm(range(225)):
    # load cellpose masks and add (FOV x 1000) to the mask values to distinguish them from other FOVs
    seg_mask = np.load(f'C:\\Users\\zgibbs\\cellpose\\20CRE_T7_July_2024\\20CRE_T7_July_2024_seg\\polyA\\Conv_zscan__{fov:03d}_seg.npy', allow_pickle=True)[()]
    mask_id = fov * 1000 
    seg_mask['masks'] = seg_mask['masks'] + mask_id 

    # specify the appropriate FOV for spots
    spots = fov_counts[fov]
    
    # filter spots to remove those outside the image dimensions following drift correction
    filtered_spots = {key: coords[(coords[:, 2] <= 2999) & (coords[:, 2] > 0) & (coords[:, 1] <= 2999) & (coords[:, 1] > 0), :] for key, coords in spots.items()}
    
    # round the x,y coordinates for decoded spots and assign to the masks from cellpose
    spot_masks = []
    for key in filtered_spots.keys():
        filtered_spots[key][:,2] = np.around(filtered_spots[key][:,2])
        filtered_spots[key][:,1] = np.around(filtered_spots[key][:,1])
        masks = seg_mask['masks'][np.around(filtered_spots[key][:,1]).astype(int), np.around(filtered_spots[key][:,2]).astype(int)]
        spot_masks += [np.array([masks, np.array([key] * (filtered_spots[key].shape[0])), np.around(filtered_spots[key][:,2]), np.around(filtered_spots[key][:,1])])]
        
    spot_masks = np.hstack(spot_masks).T
    barcodes = pd.DataFrame(data=spot_masks, columns=['masks', 'cre_id', 'x_round', 'y_round'])
    cell_by_gene = pd.crosstab(barcodes.masks, barcodes.cre_id)
    fov_means.append(cell_by_gene.mean(axis=0))
    all_cbg.append(cell_by_gene)
    
    # can save the cell-by-gene matrix for each FOV separately:
    # cell_by_gene.to_csv(f'/projects/ps-renlab2/zgibbs/STARR-FISH/8_23_2023__CRE-20_10XCREConc/cell_by_gene/cell_by_gene_{fov:03d}.csv')
    
all_cbg = pd.concat(all_cbg, axis=0)

100%|████████████████████████████████████████████████████████████████████████████████| 225/225 [01:19<00:00,  2.84it/s]


##### *Remove non-cell masks from the final cbg matrix*

In [91]:
non_cells = []

for fov in range(225):
    mask_id = fov * 1000
    non_cells.append(str(mask_id))
    
all_cbg.drop(labels=non_cells, axis=0, inplace = True)

##### *Add new columns to indicate total transcripts & the fov identity for each cell*

In [93]:
all_cbg['total transcripts'] = all_cbg.sum(axis=1)
all_cbg['fov'] = pd.cut(all_cbg.index.astype(int), bins = np.arange(0, 226)*1000, right=False)

In [94]:
all_cbg

cre_id,CRE001,CRE002,CRE003,CRE004,CRE005,CRE006,CRE007,CRE008,CRE009,CRE010,...,CRE013,CRE014,CRE015,CRE016,CRE017,CRE018,CRE019,CRE020,total transcripts,fov
masks,,,,,,,,,,,,,,,,,,,,,
1,5,7,2,1,0,1,0,1,1,0,...,3,0,0,0,1,1,0,4,28,"[0, 1000)"
10,20,22,12,6,2,1,3,7,5,3,...,7,0,5,2,8,8,2,4,123,"[0, 1000)"
11,5,11,6,5,2,1,2,7,1,1,...,8,1,1,2,1,5,2,7,70,"[0, 1000)"
12,17,18,27,5,3,0,1,4,4,0,...,8,0,4,5,3,6,2,8,117,"[0, 1000)"
13,27,40,35,12,12,5,3,12,9,3,...,13,0,9,9,6,15,4,8,236,"[0, 1000)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
224058,13,16,6,1,4,3,2,5,5,1,...,4,1,2,5,3,3,3,5,87,"[224000, 225000)"
224059,6,4,2,1,0,0,0,2,0,0,...,1,0,0,0,0,1,1,1,19,"[224000, 225000)"
224060,12,32,11,14,15,14,13,15,7,12,...,18,4,13,18,12,22,13,21,286,"[224000, 225000)"


##### *Save the cell-by-gene matrix*

In [102]:
# all_cbg.to_csv(r'C:\Users\zgibbs\cellpose\cell_by_gene\SFv4_T7_July_enhancer_cbg.csv')